In [ ]:
import pandas as pd
import numpy as np
from utils.load_data import load_raw_nmt2025
from pathlib import Path

## Load raw data

In [ ]:
data=load_raw_nmt2025()

In [ ]:
data.info()

## Subject columns

In [ ]:
subjects = [subj.replace('BlockBall100','') for subj in data.columns if 'BlockBall100' in subj]

# Scale 100-200
columns_ball100 = [f'{subj}BlockBall100' for subj in subjects]

# Standard scale
columns_ball = [f'{subj}BlockBall' for subj in subjects]

# Subject names
columns_name = [f'{subj}Block' for subj in subjects]

## Cleaning

In [ ]:
data_clean=data.copy()

In [ ]:
# Drop unnecessary columns

data_clean.drop(['outid','Test'],axis=1,inplace=True)
data_clean.drop([col for col in columns_name],axis=1,inplace=True)

Convert scores to int16 and nan to -1

In [ ]:
# Delete ,0 from a number and convert string -> int16 and NaN -> -1
for column in columns_ball100:
    data_clean[column] = data_clean[column].str.replace(',0','').fillna(-1).astype(np.int16)

In [ ]:
# Convert float -> int16 and nan -> -1
for column in columns_ball:
    data_clean[column] = data[column].fillna(-1).astype(np.int16)

## Create dataframe for geo: transliterate AreaName column

In [ ]:
data_geo=data_clean.copy()

In [ ]:
# Transliterate area names in NMT data file
translit_map = {
        'А': 'A', 'а': 'a', 'Б': 'B', 'б': 'b',
        'В': 'V', 'в': 'v', 'Г': 'H', 'г': 'h',
        'Ґ': 'G', 'ґ': 'g', 'Д': 'D', 'д': 'd',
        'Е': 'E', 'е': 'e', 'Є': 'Ye', 'є': 'ie',
        'Ж': 'Zh', 'ж': 'zh', 'З': 'Z', 'з': 'z',
        'И': 'Y', 'и': 'y', 'І': 'I', 'і': 'i',
        'Ї': 'Yi', 'ї': 'i', 'Й': 'Y', 'й': 'i',
        'К': 'K', 'к': 'k', 'Л': 'L', 'л': 'l',
        'М': 'M', 'м': 'm', 'Н': 'N', 'н': 'n',
        'О': 'O', 'о': 'o', 'П': 'P', 'п': 'p',
        'Р': 'R', 'р': 'r', 'С': 'S', 'с': 's',
        'Т': 'T', 'т': 't', 'У': 'U', 'у': 'u',
        'Ф': 'F', 'ф': 'f', 'Х': 'Kh', 'х': 'kh',
        'Ц': 'Ts', 'ц': 'ts', 'Ч': 'Ch', 'ч': 'ch',
        'Ш': 'Sh', 'ш': 'sh', 'Щ': 'Shch', 'щ': 'shch',
        'Ю': 'Yu', 'ю': 'iu', 'Я': 'Ya', 'я': 'ia',
        'Ь': '', 'ь': '', 'Ъ': '', 'ъ': '', "'": ""
    }

def transliterate_ukrainian(name: str) -> str:

    original = name.strip()

    # Remove "район"
    if original.endswith("район"):
        base = original.replace("район", "").strip()
        translit = "".join(translit_map.get(ch, ch) for ch in base)
        return translit

    # Remove "м."
    if original.startswith("м."):
        base = original[2:].strip()
        translit = "".join(translit_map.get(ch, ch) for ch in base)
        return translit

    # Default: transliterate the whole string
    return "".join(translit_map.get(ch, ch) for ch in original)


# Apply
data_geo['AreaName']=data_clean['AreaName'].apply(transliterate_ukrainian)

## Dataframe without -1

In [ ]:
# Drop rows with math, ukr and hist = -1
data_without_not_passed=data_clean[(data_clean['UkrBlockBall']!=-1) & (data_clean['MathBlockBall']!=-1) & (data_clean['HistBlockBall']!=-1)]

## Save new dataframes

In [ ]:
path_cleaned = Path.cwd().parent / 'data' / 'nmt2025_cleaned.csv'
data_clean.to_csv(path_cleaned,index=False)

In [ ]:
path_geo = Path.cwd().parent / 'data' / 'nmt2025_geo.csv'
data_geo.to_csv(path_geo,index=False)

In [ ]:
path_without_not_passed = Path.cwd().parent / 'data' / 'nmt2025_without_not_passed.csv'
data_without_not_passed.to_csv(path_without_not_passed,index=False)